# Project Notebook: Yelp Dataset Review Generator

In [ ]:
!pip install datasets

In [ ]:
# Import packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import argparse
# Use a pipeline as a high-level helper
from datasets import load_dataset
from transformers import pipeline, set_seed, GPT2Tokenizer, GPT2Model,AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments

# !pip install datasets
from datasets import load_dataset # https://huggingface.co/docs/datasets/v3.1.0/en/package_reference/main_classes#datasets.Dataset


The Yelp Reviews dataset is available on [Hugging Face](https://huggingface.co/datasets/Yelp/yelp_review_full) and can be loaded using Pandas.

### Note!

The version of this dataset which we used for Homework 2 was "polarized" and used 0/1 label encoding for positive/negative sentiment. This is the original dataset which encodes labels using integers **0 through 4** to represent the corresponding star rating of each review (1 star to 5 stars) with 4 denoting the most positive sentiment and 0 denoting the most negative.

We hope that this more-granular labeling will allow for more project possibilities; if you wish to use 0/1 labels, feel free to define another label encoding and make it explicit in your project report (for example, mapping original labels 0-2 to 0 and mapping original labels 3-4 to 1).

The dataset is divided into train and test splits with 650k and 50k examples, respectively. For a validation set, we recommend subsetting from the training dataset.

Now you're ready to explore the dataset!

In [ ]:
# HYPERPARAMETERS
batch_size = 8
num_epochs = 5

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
def tokenize_function(examples):
    # print(examples)
    inputs =  tokenizer(examples['text'], truncation=True, padding='max_length', max_length=128)
    inputs['labels'] = inputs['input_ids'].copy()
    return inputs

In [ ]:
# Loading and Tokenizing the Dataset
dataset = load_dataset("yelp_review_full")

x = 5000
dataset_train = dataset['train'].select(range(x))
dataset_test = dataset['test'].select(range(x // 10))
# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

tokenized_dataset_train = dataset_train.map(tokenize_function, batch_size = batch_size)
tokenized_dataset_test = dataset_test.map(tokenize_function, batch_size = batch_size)

# Fine tuning

This samples training and testing data from the overall dataset.

In [ ]:
# https://medium.com/@prashanth.ramanathan/fine-tuning-a-pre-trained-gpt-2-model-and-performing-inference-a-hands-on-guide-57c097a3b810

# https://huggingface.co/openai-community/gpt2#how-to-use
model = AutoModelForCausalLM.from_pretrained('gpt2').to(device)
output_dir = '/mnt/results'
logging_dir = '/mnt/logs'
model_output_dir = '/mnt/results/model'


In [ ]:
# Define training arguments
output_dir = '/mnt/results'
logging_dir = '/mnt/logs'
training_args = TrainingArguments(
    output_dir=output_dir,
    evaluation_strategy='epoch',
    num_train_epochs=num_epochs,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir=logging_dir,
    report_to="none",
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset = tokenized_dataset_train,
    eval_dataset=tokenized_dataset_test,
)

# Train the model
checkpoint = '/mnt/results/checkpoint-5000'
trainer.train(resume_from_checkpoint=checkpoint)
trainer.train()

# save the model and tokenizer explicitly
model_output_dir = '/mnt/results/model'

model.save_pretrained(model_output_dir)
tokenizer.save_pretrained(model_output_dir)

# Trying our finetuned model

In [ ]:
def makeYelpPrompt(review, starRating, sentiment="serious"):
    return f'''\
Continue the following Yelp review which gives a rating of {starRating+1} stars and has a {sentiment} tone:
{review}\
'''

In [ ]:
generator = pipeline('text-generation', model=model, tokenizer=tokenizer, device=device)
set_seed(42)

In [ ]:
prompts = []
prompts.append(makeYelpPrompt("I am so disappointed in", 0)) # Negative prompt
prompts.append(makeYelpPrompt("I am in awe of", 4)) # Positive prompt
prompts.append(makeYelpPrompt("I am in awe of", 0)) # "Positive" prompt
prompts.append(makeYelpPrompt("This restaurant is", 0)) # Can GPT fill in sentiment based on star rating?
prompts.append(makeYelpPrompt("This restaurant is", 4)) # Can GPT fill in sentiment based on star rating?
prompts.append(makeYelpPrompt("This restaurant is", 2)) # Can GPT fill in sentiment based on star rating?
prompts.append(makeYelpPrompt("This restaurant is", 2, "funny")) # Can GPT fill in tone?
prompts.append(makeYelpPrompt("This restaurant is", 2, "optimistic")) # Can GPT fill in tone?
# Are longer initial reviews better?
prompts.append(makeYelpPrompt("Rose tea is great value for the price, especially for a college student. There’s usually a 10 min wait if you order at the counter so I suggest ordering ahead of time. The fried chicken over rice is", 4))
prompts.append(makeYelpPrompt("I used to enjoy this place...until I got food poisoning. I ordered a chicken & broccoli dish and believe it was the broccoli that was somehow contaminated. Haven't been back since. If you eat something from here that doesn't taste right on first bite, trust your gut (literally)", 0))

In [ ]:
for prompt in prompts:
    output = generator(prompt, max_length=50 if len(prompt) < 50 else 120, num_return_sequences=5)
    for d in output:
        print(d['generated_text'])
        print('-----------------------')
    print('~~~~~~~~~~~~~~~~~~')

# Generating reviews from test data

In [ ]:
dataset_test_large = dataset['test'].select(range(x // 10))

In [ ]:
length = 10
test_prompts = dataset_test_large.map(lambda x: {'prompt': makeYelpPrompt(' '.join(x['text'].split()[:length]), x['label'])})
# test_prompts_large = dataset_test_large.map(lambda x: {'prompt': makeYelpPrompt(' '.join(x['text'].split()[:length]), x['label'])})

In [ ]:
output = generator(test_prompts['prompt'], max_length=75)
temp = test_prompts.add_column('output', [L[0]['generated_text'] for L in output])
test_prompts = temp.map(lambda x: {'generated_output' : ':'.join(x['output'].split(':')[1:])})
test_prompts = test_prompts.remove_columns('output')

# Using classifiers to quantify performance

In [ ]:
from transformers import pipeline
import torch.nn as nn
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# We will be using BCE loss to measure the performance of our fine-tuned transformer against these models.

loss_fn = nn.BCELoss()


This classifier, found at https://huggingface.co/nlptown/bert-base-multilingual-uncased-sentiment, is designed for product reviews. It maps an input text to a probability distribution on $\Omega=\{1,2,3,4,5\}$, which is the number of stars.

In [ ]:
pipe = pipeline("text-classification", model="nlptown/bert-base-multilingual-uncased-sentiment", device="cuda",return_all_scores=True)

one_hots = nn.functional.one_hot(torch.tensor([0,1,2,3,4]), num_classes=5).float()
# Using cross entropy loss from the 1-hot vector on expected_sentiment with the classifier's output on review
def calculate_review_loss(review, expected_sentiment):
  res = pipe(review)

  # turn result into probability distribution
  distribution = torch.tensor([x['score'] for x in res[0]])

  # alternative: use gaussian based assumption instead of one-hot
  # sigma = .5
  # class_indices = torch.arange(5, dtype=torch.float32)
  # gaussian = torch.exp(-((class_indices - (expected_sentiment)) ** 2) / (2 * sigma ** 2))
  # gaussian /= gaussian.sum()

  # return BCE loss
  return loss_fn(distribution, one_hots[expected_sentiment])

print(calculate_review_loss("Rose tea is possibly the very best thing I have ever tasted.", 4))


This classifier, found at https://huggingface.co/lxyuan/distilbert-base-multilingual-cased-sentiments-student, is designed for product reviews. It maps an input text to a probability distribution on $\Omega=\{-1,0,1\}$, which indicates negative, neutral, and positive sentiment.

Finally, this third classifier, found at https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest, was trained on tweets. It maps an input text to a probability distribution on $\Omega=\{0,1,2\}$, which indicates negative, neutral, and positive sentiment respectively.

In [ ]:
test_prompts = test_prompts.map(lambda x: {'review_loss' : calculate_review_loss(x['generated_output'], x['label'])})

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.hist((test_prompts['review_loss']), bins=30, alpha=0.6, color='b')

# Add titles and labels
plt.title("Distribution")
plt.xlabel("Binary Cross-entropy loss")
plt.ylabel("Count")

# Show the plot
plt.show()


In [ ]:
print(sum(test_prompts['review_loss']) / len(test_prompts['review_loss']))

In [ ]:
import json
with open('/content/generated_outputs.json', 'w') as f:
    json.dump(test_prompts.to_dict(), f,indent=4)